# Prepare FHRS data

In this notebook, the FHRS working dataset will be validated so that bakery classification can occur on a reduced dataset. 

Checks will include duplicate establishment IDs, missing values, postcode quality, coordinate availability, business types and business-name formatting.

In [7]:
from pathlib import Path

import pandas as pd

INTERIM_FOLDER = Path("../data/interim")

DATA_PATH = (INTERIM_FOLDER/ "london_fhrs_req_columns_2026-07-23.csv")

# Loading data
fhrs_working_data = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {fhrs_working_data.shape}")

fhrs_working_data.head()

Dataset shape: (81085, 7)


,FHRSID,BusinessName,BusinessType,PostCode,LocalAuthorityName,geocode.longitude,geocode.latitude
0,56458,Holibrook House Childrens Home,Caring Premises,IG11 8RB,Barking and Dagenham,0.081816,51.543048
1,66309,BP Service Station,Retailers - other,RM10 7UU,Barking and Dagenham,0.161954,51.550546
2,91426,Playdays Nursery and PreSchool,Caring Premises,RM8 1DD,Barking and Dagenham,0.134142,51.561604
3,91591,Co-op Welcome,Retailers - supermarkets/hypermarkets,RM9 4TP,Barking and Dagenham,0.127901,51.539792
4,95942,K.F.C.,Takeaway/sandwich shop,IG11 8EB,Barking and Dagenham,0.080525,51.539087


# Checking duplicate establishments

In [8]:
duplicate_rows = fhrs_working_data[fhrs_working_data["FHRSID"].duplicated(keep=False)].sort_values("FHRSID")

print(f"Duplicated FHRS IDs: {duplicate_rows["FHRSID"].nunique()}")

print(f"Rows involving duplicated IDs: {len(duplicate_rows)}")

print(f"Completely identical duplicate rows: {fhrs_working_data.duplicated().sum()}")

duplicate_rows

Duplicated FHRS IDs: 5
Rows involving duplicated IDs: 10
Completely identical duplicate rows: 5


,FHRSID,BusinessName,BusinessType,PostCode,LocalAuthorityName,geocode.longitude,geocode.latitude
46863,605931,Sainsbury's,Retailers - supermarkets/hypermarkets,SW10 9EW,Kensington and Chelsea,-0.185072,51.483876
46864,605931,Sainsbury's,Retailers - supermarkets/hypermarkets,SW10 9EW,Kensington and Chelsea,-0.185072,51.483876
26710,695147,Tesco,Retailers - supermarkets/hypermarkets,SE9 1DH,Greenwich,0.050239,51.451492
26711,695147,Tesco,Retailers - supermarkets/hypermarkets,SE9 1DH,Greenwich,0.050239,51.451492
57096,979272,German Doner Kebab,Takeaway/sandwich shop,E15 1NG,Newham,0.001902,51.541653
57095,979272,German Doner Kebab,Takeaway/sandwich shop,E15 1NG,Newham,0.001902,51.541653
3827,1884567,Caffe Nero,Other catering premises,NW4 3FN,Barnet,-0.224534,51.576185
3826,1884567,Caffe Nero,Other catering premises,NW4 3FN,Barnet,-0.224534,51.576185
16958,1893632,Hub By Premier Inn,Hotel/bed & breakfast/guest house,EC1A 2DP,City of London Corporation,-0.102824,51.517061
16959,1893632,Hub By Premier Inn,Hotel/bed & breakfast/guest house,EC1A 2DP,City of London Corporation,-0.102824,51.517061


## Removing exact duplicate records

In [13]:
conflicting_ids = []

for fhrs_id, group in duplicate_rows.groupby("FHRSID"):
    unique_records = group.drop_duplicates()

    if len(unique_records) > 1:
        conflicting_ids.append(fhrs_id)

print(f"Duplicated IDs with conflicting data: {len(conflicting_ids)}")
print(conflicting_ids)

fhrs_prepared = fhrs_working_data.drop_duplicates().copy()

print(f"Rows before removing duplicates: {len(fhrs_working_data)}")
print(f"Rows after removing duplicates: {len(fhrs_prepared)}")
print(f"Duplicate rows removed: {len(fhrs_working_data) - len(fhrs_prepared)}")

Duplicated IDs with conflicting data: 0
[]
Rows before removing duplicates: 81085
Rows after removing duplicates: 81080
Duplicate rows removed: 5


# Standardising business names for querying

In [31]:
fhrs_prepared["BusinessNameClean"] = (
    fhrs_prepared["BusinessName"]
    .astype("string")
    .str.casefold()
    .str.replace("’", "'", regex=False)
    .str.replace("‘", "'", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

fhrs_prepared[["BusinessName", "BusinessNameClean"]].sample(n=5, random_state=42)

,BusinessName,BusinessNameClean
64104,Belair house,belair house
45276,Aetos Greek Taverna,aetos greek taverna
61074,MNP Flavours of India,mnp flavours of india
25906,Nicholas Catering,nicholas catering
51402,A-Z Grocery,a-z grocery


# Validating prepared dataset

In this case we are mainly concerned with checking the preparation of the business names and removing duplicates.

In [32]:
print(f"Missing original names: {fhrs_prepared["BusinessName"].isna().sum()}")

print(f"Missing cleaned names: {fhrs_prepared["BusinessNameClean"].isna().sum()}")

print(f"Duplicate FHRS IDs remaining: {fhrs_prepared["FHRSID"].duplicated().sum()}")

print(f"Prepared establishment dataset shape: {fhrs_prepared.shape}")

Missing original names: 0
Missing cleaned names: 0
Duplicate FHRS IDs remaining: 0
Prepared establishment dataset shape: (81080, 8)


# Saving prepared dataset

In [33]:
PREPARED_PATH = (INTERIM_FOLDER/ "london_fhrs_prepared_2026-07-23.csv")

fhrs_prepared.to_csv(PREPARED_PATH, index=False)

print(f"Prepared establishments saved to: {PREPARED_PATH}")

Prepared establishments saved to: ..\data\interim\london_fhrs_prepared_2026-07-23.csv
